# Data Features Selection
This notebook is the base for features  selection
## Used libraries

In [12]:
# type: ignore
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE, SelectKBest, SelectFromModel, f_classif, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA 
from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import cross_val_score 

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from statsmodels.stats.multitest import multipletests

## Loading data

The train and test inputs are composed of 46 features.

The target of this challenge is `RET` and corresponds to the fact that the **return is in the top 50% of highest stock returns**.

Since the median is very close to 0, this information should not change much with the idea to predict the sign of the return.

In [ ]:
train = pd.read_csv('../train_extended.csv', index_col='ID')
test = pd.read_csv('../test_extended.csv', index_col='ID')

In [5]:
target = 'RET'
x_train, y_train = train.drop(target, axis=1), train[target]

In [6]:
# Infinites values per column
infinite_train = x_train.isin([np.inf, -np.inf]).sum()
infinite_train = infinite_train[infinite_train > 0]
infinite_test = test.isin([np.inf, -np.inf]).sum()
infinite_test = infinite_test[infinite_test > 0]


In [7]:
# replace the columns with infinite values
x_train = x_train.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)


In [ ]:
# Missing values per column
missing = train.isnull().sum()
missing = missing[missing > 0]
missing = missing.sort_values(ascending=False)
missing


RSI_5_SECTOR_DATE                  11071
RSI_10_SECTOR_DATE                   563
VOLUME_5_SUB_INDUSTRY_DATE_skew      536
VOLUME_5_SUB_INDUSTRY_DATE_kurt      536
VOLUME_4_SUB_INDUSTRY_DATE_skew      518
                                   ...  
VOLUME_2_SECTOR_DATE_std               1
VOLUME_1_SECTOR_DATE_kurt              1
VOLUME_1_SECTOR_DATE_skew              1
VOLUME_1_SECTOR_DATE_std               1
Corr_RET_VOL_20                        1
Length: 139, dtype: int64

In [ ]:
# plot correlation matrix of features
plt.figure(figsize=(12, 10))
cor = x_train.corr()
sns.heatmap(cor, annot=False, cmap=plt.cm.Reds)
plt.show()

In [ ]:

corr_matrix = np.corrcoef(x_train, rowvar=False)
high_corr = np.where(np.abs(corr_matrix) > 0.8)

In [ ]:
# Liste des colonnes à supprimer
cols_to_remove = set([i for i, j in zip(*high_corr) if i != j])
cols_to_remove = list(cols_to_remove)

In [ ]:
features_to_remove = x_train.columns[cols_to_remove]
features_to_keep = x_train.columns.difference(features_to_remove)
features_to_keep

In [ ]:
# correlation matrix after removing highly correlated features
plt.figure(figsize=(12, 10))
cor = x_train[features_to_keep].corr().abs()
sns.heatmap(cor, annot=False, cmap=plt.cm.Reds)
plt.show()


## Feature selection

In [ ]:
n_shifts = 5  # If you don't want all the shifts to reduce noise
features = ['RET_%d' % (i + 1) for i in range(n_shifts)]
features += ['VOLUME_%d' % (i + 1) for i in range(n_shifts)]
features += features_to_keep.tolist()
#features += cat_features  # The categorical features if we want to use them
train[features].head()

In [ ]:
# Feature selection through prefit model and SelectFromModel
model = RandomForestClassifier(n_estimators=50, max_depth=6,  n_jobs=-1, verbose=1)
model.fit(x_train[features], y_train)

# feature importance of the model
importances = model.feature_importances_

# plot the feature importances sorted
indices = np.argsort(importances)[::-1]
sns.barplot(x=importances[indices], y=x_train[features].columns[indices], orient='h')

In [ ]:
# save the features importance of the model ranked by importance with their names
features_importance = pd.DataFrame({'feature': x_train.columns, 'importance': model.feature_importances_})
features_importance = features_importance.sort_values(by='importance', ascending=False)
features_importance.to_csv('features_importance.csv')

In [ ]:
# Select 40 features with SelectFromModel with the prefit model
pre_selector = SelectFromModel(model, max_features=50, prefit=True)
pre_selectedFeatures = x_train.columns[pre_selector.get_support()]

pre_selectedFeatures

In [ ]:
# correlation matrix of the selected features
plt.figure(figsize=(12, 10))
cor = train[list(pre_selectedFeatures) + ['RET']].corr().abs()
sns.heatmap(cor, annot=False, cmap=plt.cm.Reds)
plt.show()


In [ ]:
# Procedding to RFE forward selection from the selected features
# We will use the RandomForestClassifier as estimator
estimator = XGBClassifier(n_estimators=500, max_depth=8, n_jobs=-1)
selector = RFE(estimator, n_features_to_select=45, step=1, verbose=10)
selector.fit(x_train[pre_selectedFeatures], y_train)
selectedFeaturesXGB = pre_selectedFeatures[selector.support_]  # Get the selected features

In [ ]:
estimator2 = RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1)
selector2 = RFE(estimator2, n_features_to_select=45, step=1, verbose=10)
selector2.fit(x_train[pre_selectedFeatures], y_train)
selectedFeaturesRF = pre_selectedFeatures[selector2.support_]  # Get the selected features

In [ ]:
# save the selected features of the RFE XGB and RF by transforming them to a DataFrame then save them to a csv file
selectedFeaturesXGB = pd.DataFrame(selectedFeaturesXGB, columns=['feature'])
selectedFeaturesRF = pd.DataFrame(selectedFeaturesRF, columns=['feature'])
selectedFeaturesXGB.to_csv('selectedFeaturesXGB.csv')
selectedFeaturesRF.to_csv('selectedFeaturesRF.csv')

In [ ]:
# Selection of features with SelectKBest f_classif
pipe_selection = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('selector', SelectKBest(score_func=f_classif, k='all'))
])

pipe_selection.fit(x_train, y_train)
scores = pipe_selection.named_steps['selector'].scores_

# put the scores in a DataFrame
scores_f = pd.DataFrame({'feature': x_train.columns, 'score': scores})
scores_f = scores_f.sort_values(by='score', ascending=False)
scores_f.to_csv('../scores_f.csv')


In [ ]:
# Selection of fezatures with SelectKBest mutual_info_classif
pipe_selection = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('selector', SelectKBest(score_func=mutual_info_classif, k='all'))
])

pipe_selection.fit(x_train, y_train)
scores_mutual_info = pipe_selection.named_steps['selector'].scores_

# put the scores in a DataFrame
scores_mutual_info_f = pd.DataFrame({'feature': x_train.columns, 'score': scores_mutual_info})
scores_mutual_info_f = scores_mutual_info_f.sort_values(by='score', ascending=False)
scores_mutual_info_f.to_csv('../scores_mutual_info.csv')

In [ ]:
# PCA Pipeline on the whole dataset

pipe_pca = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.99))
])

pipe_pca.fit(x_train)

# draw the explained variance ratio of the PCA
plt.plot(pipe_pca.named_steps['pca'].explained_variance_ratio_)
plt.xlabel('Number of components')
plt.ylabel('Explained variance ratio')
plt.show()


In [ ]:
# train the model with the PCA selected features

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=45)),
    ('model', RandomForestClassifier(n_estimators=500, max_depth=8, n_jobs=-1))
])

scores = cross_val_score(pipe, x_train, y_train, cv=3, scoring='accuracy', verbose=10)
for i, score in enumerate(scores):
    print('Fold %d: %.3f' % (i, score))
print('Accuracy: %.3f +/- %.3f' % (scores.mean(), scores.std()))


In [ ]:

X = x_train

# 🔹 Suppose que X est un DataFrame Pandas avec les features
fixed_features = pd.read_csv('../features_fixed.csv', index_col=0) # Features qu'on garde toujours
fixed_features = list(fixed_features['0'])
fixed_features[-1] = 'ADL_5'  # On corrige un bug
candidate_features = [col for col in X.columns if col not in fixed_features]  # Features à tester


In [ ]:

# 🔹 Séparer les données
X_fixed = x_train[fixed_features]  # Features fixes (on les garde toujours)
X_candidates = x_train[candidate_features]  # Features à tester

# 🔹 Appliquer SFS sur les "features candidates"
 
sfs = SequentialFeatureSelector(
    RandomForestClassifier(n_estimators=50, max_depth=6, n_jobs=-1, verbose=3),
    n_features_to_select=1,  # On ajoute 1 features aux features fixes
    direction="forward",
    cv=3,
    n_jobs=-1,
)

sfs.fit(X_candidates, y_train)

print("Features fixes :", fixed_features)
print("Features sélectionnées :", list(X_candidates.columns[sfs.get_support()]))


In [ ]:

def feature_selection_classification(df, target_col, mi_threshold=0.01, pval_threshold=0.05, corr_threshold=0.9):
    """
    Selects the most relevant features for binary classification using:
    1. Mutual Information (MI) to detect nonlinear relationships.
    2. F-Test to compute p-values for statistical significance.
    3. False Discovery Rate (FDR) correction for multiple comparisons.
    4. Correlation filtering to remove redundant features.

    Parameters:
    - df (pd.DataFrame): Dataset containing features and target.
    - target_col (str): Name of the target variable (binary: 0 or 1).
    - mi_threshold (float): Minimum MI score for feature relevance.
    - pval_threshold (float): Maximum corrected p-value for feature selection.
    - corr_threshold (float): Maximum allowed correlation (default 0.9).

    Returns:
    - selected_features (list): List of chosen feature names.
    - df_selected (pd.DataFrame): New dataframe with selected features.
    """

    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Compute Mutual Information for classification
    mi_scores = mutual_info_classif(X, y)
    mi_scores = pd.Series(mi_scores, index=X.columns)
    
    # Compute F-test p-values
    f_scores, p_values = f_classif(X, y)
    p_values = pd.Series(p_values, index=X.columns)
    
    # Apply False Discovery Rate (FDR) correction for multiple comparisons
    _, p_values_corrected, _, _ = multipletests(p_values, alpha=pval_threshold, method='fdr_bh')
    p_values_corrected = pd.Series(p_values_corrected, index=X.columns)

    # Step 1: Select features based on MI and p-values
    selected_features = mi_scores[(mi_scores > mi_threshold) & (p_values_corrected < pval_threshold)].index.tolist()

    # Step 2: Remove highly correlated features
    df_selected = df[selected_features + [target_col]]  # Keep only selected features + target
    corr_matrix = df_selected.corr().abs()

    # Identify highly correlated pairs
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > corr_threshold)]

    # Final feature list after removing correlated features
    selected_features = [f for f in selected_features if f not in to_drop]

    # Return final dataframe with selected features
    return selected_features, df[selected_features + [target_col]]

# Example Usage:
# selected_feats, x_train_selected = feature_selection_classification(x_train, target_col="TARGET")
# print("Selected Features:", selected_feats)
